# Topic 8 — Random Forest

> **Goal of this notebook:** understand how **bagging** many decorrelated decision trees into a forest turns a high-variance model into a robust, accurate one. We'll see the out-of-bag estimate, feature importance, and build a mini-forest from scratch.

**Contents**
1. Concept & intuition
2. How it works (bagging, feature randomness, OOB)
3. Key hyperparameters
4. The dataset: Breast Cancer
5. Random Forest vs a single tree
6. Out-of-bag score & number of trees
7. Feature importance
8. A mini bagging ensemble from scratch
9. Pros, cons & when to use
10. Summary

## 1. Concept & Intuition

A single decision tree is accurate but **unstable** — it has high variance. A **Random Forest** averages many trees so their individual errors cancel out, like asking a large, diverse crowd instead of one expert.

The trick is making the trees **different from each other** (decorrelated). If all trees were identical, averaging would change nothing. Random Forest injects two sources of randomness to achieve that.

## 2. How It Works

1. **Bagging (Bootstrap AGGregatING):** each tree is trained on a random sample of the data drawn *with replacement* (a bootstrap sample, ~63% unique rows).
2. **Feature randomness:** at each split, the tree may only choose from a random subset of features (`max_features`). This stops one strong feature from dominating every tree.
3. **Aggregation:** classification → majority vote across trees; regression → average.

**Out-of-bag (OOB) score:** the ~37% of rows each tree *didn't* see act as a free validation set, giving an unbiased accuracy estimate without a separate holdout.

Averaging reduces **variance** without increasing bias — that's why the forest beats its trees.

## 3. Key Hyperparameters

| Parameter | Effect |
|-----------|--------|
| **n_estimators** | number of trees — more is better (diminishing returns), never overfits by adding trees. |
| **max_features** | features considered per split — lower = more decorrelation. |
| **max_depth / min_samples_leaf** | size of each tree. |
| **oob_score** | enable the out-of-bag estimate. |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Dataset: Breast Cancer

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 5. Random Forest vs a Single Tree

In [ ]:
single = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=300, oob_score=True,
                                random_state=42, n_jobs=-1).fit(X_train, y_train)

print('Single tree  test accuracy:', round(single.score(X_test, y_test), 4))
print('Random Forest test accuracy:', round(forest.score(X_test, y_test), 4))
print('Random Forest OOB accuracy: ', round(forest.oob_score_, 4))

The forest beats the single tree, and its **OOB estimate** closely tracks the test accuracy — a built-in, free validation.

## 6. Out-of-Bag Score vs Number of Trees

Accuracy rises then plateaus as we add trees — adding more never hurts, it just stops helping.

In [ ]:
ns = [1, 5, 10, 25, 50, 100, 200, 400]
oob = []
for n in ns:
    rf = RandomForestClassifier(n_estimators=n, oob_score=True, bootstrap=True,
                                random_state=42, n_jobs=-1).fit(X_train, y_train)
    oob.append(rf.oob_score_)

plt.plot(ns, oob, marker='o')
plt.xlabel('number of trees'); plt.ylabel('OOB accuracy')
plt.title('More trees → lower variance, then plateau'); plt.show()

## 7. Feature Importance

In [ ]:
imp = forest.feature_importances_
order = np.argsort(imp)[-10:]
plt.barh([data.feature_names[i] for i in order], imp[order])
plt.xlabel('importance'); plt.title('Top 10 features (Random Forest)'); plt.show()

## 8. A Mini Bagging Ensemble From Scratch

Train many trees on bootstrap samples and majority-vote — the essence of a forest.

In [ ]:
class BaggingScratch:
    def __init__(self, n_trees=100, max_features='sqrt'):
        self.n_trees, self.max_features = n_trees, max_features
    def fit(self, X, y):
        self.trees = []
        n = len(X)
        for i in range(self.n_trees):
            idx = np.random.choice(n, n, replace=True)  # bootstrap sample
            t = DecisionTreeClassifier(max_features=self.max_features, random_state=i)
            t.fit(X[idx], y[idx])
            self.trees.append(t)
        return self
    def predict(self, X):
        preds = np.array([t.predict(X) for t in self.trees])
        return stats.mode(preds, axis=0, keepdims=False).mode

bag = BaggingScratch(n_trees=200).fit(X_train, y_train)
print('From-scratch bagging accuracy:', round(accuracy_score(y_test, bag.predict(X_test)), 4))
print('scikit-learn forest accuracy: ', round(forest.score(X_test, y_test), 4))

## 9. Pros, Cons & When to Use

**Pros**
- High accuracy with little tuning; a superb default model.
- Resistant to overfitting (averaging reduces variance).
- Free **OOB** validation and trustworthy **feature importance**.
- Handles mixed feature types, needs no scaling.

**Cons**
- Larger and slower to predict than one tree; less interpretable than a single tree.
- Can struggle with very high-dimensional sparse data (e.g. text) vs linear models.
- Biased feature importance toward high-cardinality features.

**When to use**
- A reliable go-to for tabular classification/regression.
- When you want strong accuracy out-of-the-box with minimal preprocessing.

## 10. Summary

- Random Forest = **bagging** many **decorrelated** trees (bootstrap rows + random features), then voting/averaging.
- This cuts variance, so the forest beats any single tree — confirmed on Breast Cancer.
- The **OOB score** gives free validation; **feature importance** explains the model.
- Our from-scratch bagging ensemble reached forest-level accuracy, proving the mechanism.